In [1]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

In [2]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

In [3]:
# def test_connection():
#     with driver.session() as session:
#         session.run("MATCH (n) RETURN n LIMIT 1")

# try:
#     test_connection()
#     print("连接成功")
# except Exception as e:
#     print("连接失败",e)
# finally:
#     driver.close()        

In [4]:
with driver.session() as session:
    session.run("""
                create (p1:Person{name:'zhangsan',age:30,city:'beijing'}),
                (p2:Person {name:'lisi',age:23,city:'shanghai'})
                """)
    
    session.run("""
                create (c1:Company {name:'shuzhiweilai',industry:'Technology',location:'beijing'}),
                (c2:Company {name:'naixue',industry:'Education',location:'beijing'})
                """)

In [5]:
with driver.session() as session:
    p1=session.run("""
                MATCH (p:Person{name:'zhangsan'})
                set p:Student
                return p
                """)
    p2=session.run("""
                match (p:Person{name:'lisi'})
                set p:Student
                return p
                """)
    print(p1)
    print(p2)

In [6]:
with driver.session() as session:
    session.run("""
                match (p:Person {name:'zhangsan'})
                set p.age=31
                """)

In [7]:
with driver.session() as session:
    session.run("""
                match (p:Person{name:'zhangsan'})
                match (c:Company{name:'shuzhiweilai'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)
    
    session.run("""
                match(p:Person{name:'lisi'})
                match(c:Company{name:'naixue'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)

In [ ]:
import re


def neo4j_query_examples(driver,query_type,params=None):
    """
    执行各种Neo4j查询示例
    参数：
    - driver: Neo4j驱动实例
    - query_type: 查询类型，可选值包括：
    'all_persons', 'all_companies', 'filter_by_city', 'all_relationships',
    'specific_relationship', 'node_relationships', 'path_query', 'aggregation',
    'group_by', 'colleagues', 'complex_query', 'param_query', 'subgraph', 'community'
    - params: 查询参数字典，根据查询类型不同而不同
    返回：
    - 查询结果列表
    """
    if params is None:
        params={}
    results=[]
    
    with driver.session() as session:
        if query_type=='all_persons':
            result=session.run("""
            match (p:Person)
            return p.name as name, p.age as age,p.city as city
                               """)
            print("所有person节点：")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']},城市：{record['city']}")
                results.append({
                    'name':record['name'],
                    'age':record['age'],
                    'city':record['city']
                })
        elif query_type=='all_companies':
            result=session.run("""
                               match (c:Company)
                               return c.name as name,c.industry as industry,c.location as location
                               """)
            print("所有company节点")
            for record in result:
                print(f"公司：{record['name']},行业：{record['industry']},位置：{record['location']}")
                results.append({
                    'name':record['name'],
                    'industry':record['industry'],
                    'location':record['location']
                })
        elif query_type=='filter_by_city':
            city=params.get('city','beijing')
            result=session.run("""
                               match (p:Person)
                               where p.city=$city
                               return p.name as name,p.age as age
                               """,{'city':city})
            print(f"{city}人员")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']}")
                results.append({
                    'name':record['name'],
                    'age':record['age']
                })
        elif query_type=='all_relationships':
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               return p.name as person, type(r) as relationship, c.name as company
                               """)
            print("所有人员与公司的关系：")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='specific_Community':
            rel_type=params.get('rel_type','EMPLOYED_BY')
            result=session.run("""
                               match (p:Person)-[:{rel_type}]->(c:Company)
                               return p.name as person, c.name as company
                               """)
            print(f"{rel_type}关系")
            for record in result:
                print(f"{record['person']} 与 {record['company']} 有 {rel_type}关系")
                results.append({
                    'person':record['person'],
                    'company':record['company']
                })
        elif query_type=='node_Communitys':
            person_name=params.get('person_name','zhangsan')
            result=session.run("""
                               match (p:Person{name:$name})-[r]->(c)
                               return type(r) as Community,c.name as connected_to,labels(c) as node_type
                               """,{'name':person_name})
            print(f"{person_name}的所有关系")
            for record in result:
                print(f"关系类型：{record['Community']},连接到: {record['connected_to']},节点类型：{record['node_type']}")
                results.append({
                    'Community':record['relationship'],
                    'connected_to':record['connected_to'],
                    'node_type':record['node_type']
                })
        elif query_type=='aggregation':
            result=session.run("""
                               match (p:Person)-[:EMPLOYED_BY]->(c:Company)
                               return c.name as company,count(p) as employee_count,avg(p.age) as avg_age
                               """)
            print("公司员工统计：")
            for record in result:
                print(f"公司：{record['company']},员工数：{record['employee_count']},平均年龄：{round(record['avg_age'],1)}")
                results.append({
                    'company':record['company'],
                    'employee_count':record['employee_count'],
                    'avg_age':record['avg_age'],
                })
        elif query_type=='complex_query':
            min_age=params.get('min_age',25)
            location=params.get('location','beijing')
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location=$location
                               and (type(r)='EMPLOYED_BY' or type(r)='INVESTED_IN')
                               return p.name as person, p.age as age,
                                type(r) as relationship,c.name as company
                                order by p.age desc
                               """,{'min_age':min_age,'location':location})
            
            print(f"{min_age}岁以上且与{location}公司有雇佣或投资关系的人：")
            for record in result:
                print(f"{record['person']} ({record['age']}岁) {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'age':record['age'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='param_query':
            query_params={
                'min_age':params.get('min_age',25),
                'location':params.get('location','beijing'),
                'relationship_types':params.get('relationship_types',['EMPLOYED_BY','INVESTED_BY'])
            }
            
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location= $location
                               and type(r) in $relationship_types
                               return p.name as person, type(r) as relationship, c.name as company
                               
                               """,query_params)
            
            print(f"参数化查询结果（年龄》{query_params['min_age']}, 位置{query_params['location']}:")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })      
        else:
            print(f"未知的查询类型：{query_type}")
            results.append({'error':f"未知的查询类型：{query_type}"})
    return results

In [9]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

try:
    neo4j_query_examples(driver,'all_persons')
    
    neo4j_query_examples(driver,'filter_by_city',{'city':'beijing'})
    neo4j_query_examples(driver,'node_relationships',{'person_name':'zhangsan'})
    neo4j_query_examples(driver,'complex_query',{'min_age':20,'location':'beijing'})
finally:
    driver.close()

所有person节点：
姓名：zhangsan,年龄：31,城市：beijing
姓名：lisi,年龄：23,城市：shanghai
beijing人员
姓名：zhangsan,年龄：31
zhangsan的所有关系
关系类型：EMPLOYED_BY,连接到: shuzhiweilai,节点类型：['Company']
20岁以上且与beijing公司有雇佣或投资关系的人：
zhangsan (31岁) EMPLOYED_BY shuzhiweilai
lisi (23岁) EMPLOYED_BY naixue


In [10]:
import time
import pandas as pd
import concurrent.futures
from neo4j import GraphDatabase

def parallel_batched_import(statement,df,batch_size=100,max_workers=8):
    total=len(df)
    batches=(total+batch_size-1)//batch_size
    start_time=time.time()
    results=[]

    print(f"开始并行导入 {total} 行数据，分为{batches} 个批次，每批{batch_size} 条")

    def process_batch(batch_idx):
        """
    批处理函数，用于处理每个批次的数据
        """

        start=batch_idx*batch_size
        end=min(start+batch_size,total)
        batch=df.iloc[start:end]

        batch_start_time=time.time()
        try:
            with driver.session() as session:
                # UNWIND 是Cypher查询语言中的一个关键字，用于将一个列表展开为多行，$row是一个参数，表示将要传入的行数据
                # 完整意思是 将$row 参数(一个列表)中的每个元素展开，每个元素被赋值给变量value,对每个value执行后续的Cypher语句
                # id name age
                # 0 张三 24
                #1 李四 23
                # 调用.to_dict("record")后，得到
                # [{"id":1,"name":"zhangsan","age":25}]
                result=session.run(
                    "UNWIND $row as value "+statement,row=batch.to_dict("records"))
                summary=result.consume()
                batch_duration=time.time()-batch_start_time

                return {
                    "batch":batch_idx,
                    "rows":end-start,
                    "success":True,
                    "duration":batch_duration,
                    "counters":summary.counters
                }
        except Exception as e:
            batch_duration=time.time()-batch_start_time
            print(f"批次{batch_idx} （行{start}-{end-1}） 处理失败：{str(e)}")

            return {
                    "batch":batch_idx,
                    "rows":end-start,
                    "success":False,
                    "duration":batch_duration,
                    "error":str(e)
                }
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures=[executor.submit(process_batch,i) for i in range(batches)]

        # 处理完成的批次
        for i,future in enumerate(concurrent.futures.as_completed(futures)):
            result=future.result()
            results.append(result)

            if result['success']:
                print(f"批次 {result['batch']} 完成：{result['rows']} 行，耗时：{result['duration']:.2f}秒")
            else:
                print(f"批次 {result['batch']} 失败：{result['rows']} 行，耗时：{result['duration']:.2f}秒，错误：{result.get('error')}")
            # 显示进度
            print(f"进度：{i+1}/{batches} 批次完成 ({((i+1)/batches*100):.1f}%)")

    # 统计结果
    successful_rows=sum(r['rows'] for r in results if r['success'])
    failed_rows=sum(r['rows'] for r in results if not r['success'])

    duration=time.time()-start_time
    rows_per_second=successful_rows/duration if duration >0 else 0

    print(f"导入完成：总计 {total} 行，成功{successful_rows} 行，失败{failed_rows}行")
    print(f"总耗时：{duration:.2f}秒，平均速度:{rows_per_second:.2f}行/秒")
    
    return {
        "total_rows":total,
        "successful_rows":successful_rows,
        "failed_rows":failed_rows,
        "duration_seconds":duration,
        "rows_per_second":rows_per_second,
        "batch_results":results
    }
            

In [11]:
import pandas as pd
from tabulate import tabulate

df_documents = pd.read_parquet('./output/documents.parquet')

# 将数组类型的列转为字符串
df_display = df_documents.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+-----------------+----------------------+----------------------+---------------------+----------+
| id                   | human_readable_id | title           | text                 | text_unit_ids        | creation_date       | raw_data |
+----------------------+-------------------+-----------------+----------------------+----------------------+---------------------+----------+
| 3efa5357dc6c8f0f6262 | 0                 | companyInfo.txt | 华为：从通信设备到智 | ['3755a97bbf1d29f73d | 2026-07-07 20:33:38 |          |
| 609ec584ec8aa64ee766 |                   |                 | 能生态领导者         | a6033a6fc991e50fd5a2 | +0800               |          |
| 4e6f75846bd83bd194d1 |                   |                 | 华为技术有限公司自诞 | ea859a8f37dd7f98fbb4 |                     |          |
| fefc41f657a18ca29f26 |                   |                 | 生于1987年在中国深圳 | 857cceb7f006e1d4c9bd |                     |          |
| 5e5db8d1a8245a02f23e |                

In [12]:
def create_document_nodes(df_documents):
    with driver.session() as session:
        try:
            session.run("create constraint if not exists for (d:__Document__) require d.id is unique")
        except Exception as e:
            print(f"创建约束时出错 (可能已存在): (e)")

    cypher_statement="""
    merge(d:__Document__{id:value.id})
    on create set
        d.human_readable_id=value.human_readable_id,
        d.title=value.title,
        d.text=value.text,
        d.creation_date=value.creation_date,
        d.import_timestamp=timestamp()
    """

    return parallel_batched_import(cypher_statement,df_documents)

In [13]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

create_document_nodes(df_documents)

开始并行导入 1 行数据，分为1 个批次，每批100 条
批次 0 完成：1 行，耗时：0.02秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 1 行，成功1 行，失败0行
总耗时：0.02秒，平均速度:55.00行/秒


{'total_rows': 1,
 'successful_rows': 1,
 'failed_rows': 0,
 'duration_seconds': 0.018182039260864258,
 'rows_per_second': 54.999331244017256,
 'batch_results': [{'batch': 0,
   'rows': 1,
   'success': True,
   'duration': 0.01565861701965332,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 1, 'nodes-created': 1, 'properties-set': 6})}]}

In [14]:
import pandas as pd
from tabulate import tabulate

text_units = pd.read_parquet('./output/text_units.parquet')

# 将数组类型的列转为字符串
df_display = text_units.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+----------------------+----------+----------------------+----------------------+----------------------+---------------+
| id                   | human_readable_id | text                 | n_tokens | document_id          | entity_ids           | relationship_ids     | covariate_ids |
+----------------------+-------------------+----------------------+----------+----------------------+----------------------+----------------------+---------------+
| 3755a97bbf1d29f73da6 | 0                 | 华为：从通信设备到智 | 300      | 3efa5357dc6c8f0f6262 | ['d48c208c-de2f-4bd2 | ['48f869c3-145d-44f1 | []            |
| 033a6fc991e50fd5a2ea |                   | 能生态领导者         |          | 609ec584ec8aa64ee766 | -a22a-171f6c509ed4'  | -8656-7e654f18226f'  |               |
| 859a8f37dd7f98fbb485 |                   | 华为技术有限公司自诞 |          | 4e6f75846bd83bd194d1 |  '3baf7c25-14dc-4552 |  '44b4fa51-a13d-49e5 |               |
| 7cceb7f006e1d4c9bdc0 |                  

In [15]:
def setup_chunk_constraints():
    """"创建chunk标签的约束 """
    with driver.session() as session:
        try:
            # 创建chunk.id唯一约束
            session.run("create constraint if not exists for (c:__Chunk__) require c.id is unique")
            print("已创建Chunk.id 唯一约束")
        except Exception as e:
            print(f"创建__Chunk__约束时出错（可能已存在）: {e}")
            #尝试旧版本Neo4j的语法
            try:
                session.run("create constraint on (c:__Chunk__) assert c.id is unique")
                print("已使用旧语法创建__Chunk__.id唯一约束")
            except Exception as e2:
                print(f"使用旧语法创建约束也失败：{e2}")

In [16]:
def import_chunks(df_chunks,batch_size=100,max_workers=8):
    """
    导入文档快(Chunk)到Neo4j
    
    参数：
    - df_chunks: 包含文档快数据的DataFrame
    - batch_size: 每批处理的行数
    - max_workers: 并行线程数
    
    返回：
    - 导入统计信息的字典
    """
    # 1. 创建chunk的约束
    setup_chunk_constraints()
    
    # 2. 创建chunk节点
    print("开始导入chunk节点。。")
    chunk_statement="""
        merge (c:__Chunk__{id:value.id})
        set c.text=value.text,
        c.n_tokens=value.n_tokens,
        c.human_readable_id=value.human_readable_id,
        c.name=value.human_readable_id
    """
    
    chunk_result=parallel_batched_import(chunk_statement,df_chunks,batch_size,max_workers)
    
    # 3. 准备关系数据 - 正确处理嵌套的document_ids并过滤无效数据
    print("准备Chunk-Document 关系数据")
    
    relations_data=[]
    
    for idx,row in df_chunks.iterrows():
        chunk_id=row['id']
        doc_ids_container=row['document_id']
        
        # 处理嵌套结构
        flat_doc_ids=[]
        
        # 如果是列表，展平它
        if isinstance(doc_ids_container,list):
            for item in doc_ids_container:
                #如果是numpy数组
                if hasattr(item,'dtype') and hasattr(item,'tolist'):
                    flat_doc_ids.extend(item.tolist())
                # 如果是列表
                elif isinstance(item,list):
                    flat_doc_ids.extend(item)
                # 如果是单个值
                else:
                    flat_doc_ids.append(item)
        # 如果不是列表直接添加
        elif doc_ids_container is not None:
            flat_doc_ids.append(doc_ids_container)
            
        # 记录处理后的document_ids
        if idx < 5: # 只记录前几行，避免日志过大
            print(f"行{idx} 处理后的document_ids:{flat_doc_ids}")
        
        #为每个文档id创建关系，过滤掉无效的ID
        for doc_id in flat_doc_ids:
            if doc_id is not None and str(doc_id).strip() !='':
                #确保doc_id是字符串并过滤掉特殊格式
                doc_id_str=str(doc_id).strip()
                
                #过滤掉不符合预期格式的id
                if doc_id_str.startswith('<elementId>') or doc_id_str.startswith('<id>'):
                    print(f"跳过无效的document_id: {doc_id_str}")
                    continue
                # 如果是列表形式的字符串，提取实际ID
                if doc_id_str.startswith('[') and doc_id_str.endswith(']'):
                    try:
                        # 尝试解析列表字符串
                        import ast
                        id_list=ast.literal_eval(doc_id_str)
                        if isinstance(id_list,list) and len(id_list) >0:
                            doc_id_str=str(id_list[0])
                    except:
                        print(f"无法解析列表格式的document_id: {doc_id_str}")
                        continue
                    
                relations_data.append({
                    'chunk_id':chunk_id,
                    'document_id':doc_id_str
                })
    # 确保Document节点存在            
    if relations_data:
        unique_doc_ids=set(item['document_id'] for item in relations_data)
        print(f"发现 {len(unique_doc_ids)} 个唯一的document_id")
        
        #创建Document节点
        docs_df=pd.DataFrame({'id':list(unique_doc_ids)})
        document_statement="""
        merge (d:__Document__{id:value.id})
        //设置显示名称
        set d.name=value.id
        """
        
        doc_result=parallel_batched_import(document_statement,docs_df,batch_size,max_workers)
        print(f"已创建 {doc_result['successful_rows']}个Document节点")
        
        # 5.创建关系
        print(f"开始创建 {len(relations_data)} 个Chunk-Document关系")
        df_relations=pd.DataFrame(relations_data)
        
        relation_statement="""
            match (c:__Chunk__{id:value.chunk_id})
            match (d:__Document__{id:value.document_id})
            merge (c)-[:PART_OF]->(d)
        """
        
        relation_result=parallel_batched_import(relation_statement,df_relations,batch_size,max_workers)
        print(f"已创建 {relation_result['successful_rows']} 个Chunk-Document关系")
    else:
        print("没有找到有效的Chunk-Document关系数据")
        
    # 6. 验证结果
    with driver.session() as session:
        # 检查chunk节点数量
        result=session.run("match (c:__Chunk__) return count(c) as count")
        chunk_count=result.single()['count']
        
        # 检查document节点数量
        result1=session.run("match (d:__Document__) return count(d) as count")
        document_count=result1.single()['count']
        
        # 检查PART_OF关系数量
        result2=session.run("match ()-[r:PART_OF]->() return count(r) as count")
        relation_count=result2.single()['count']
        
        print(f"验证结果： {chunk_count}个chunk节点，{document_count}个document节点，{relation_count}个PART_OF关系")
        
    return chunk_result

In [17]:
import_chunks(text_units)

已创建Chunk.id 唯一约束
开始导入chunk节点。。
开始并行导入 3 行数据，分为1 个批次，每批100 条
批次 0 完成：3 行，耗时：0.01秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 3 行，成功3 行，失败0行
总耗时：0.01秒，平均速度:216.46行/秒
准备Chunk-Document 关系数据
行0 处理后的document_ids:['3efa5357dc6c8f0f6262609ec584ec8aa64ee7664e6f75846bd83bd194d1fefc41f657a18ca29f265e5db8d1a8245a02f23ede293d7048b43b8c79e99fc43ad0']
行1 处理后的document_ids:['3efa5357dc6c8f0f6262609ec584ec8aa64ee7664e6f75846bd83bd194d1fefc41f657a18ca29f265e5db8d1a8245a02f23ede293d7048b43b8c79e99fc43ad0']
行2 处理后的document_ids:['3efa5357dc6c8f0f6262609ec584ec8aa64ee7664e6f75846bd83bd194d1fefc41f657a18ca29f265e5db8d1a8245a02f23ede293d7048b43b8c79e99fc43ad0']
发现 1 个唯一的document_id
开始并行导入 1 行数据，分为1 个批次，每批100 条
批次 0 完成：1 行，耗时：0.03秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 1 行，成功1 行，失败0行
总耗时：0.03秒，平均速度:35.36行/秒
已创建 1个Document节点
开始创建 3 个Chunk-Document关系
开始并行导入 3 行数据，分为1 个批次，每批100 条
批次 0 完成：3 行，耗时：0.05秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 3 行，成功3 行，失败0行
总耗时：0.05秒，平均速度:54.88行/秒
已创建 3 个Chunk-Document关系
验证结果： 3个chunk节点，1个document节点，3个PART_OF关系


{'total_rows': 3,
 'successful_rows': 3,
 'failed_rows': 0,
 'duration_seconds': 0.013859272003173828,
 'rows_per_second': 216.46158610012043,
 'batch_results': [{'batch': 0,
   'rows': 3,
   'success': True,
   'duration': 0.012329816818237305,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 3, 'nodes-created': 3, 'properties-set': 15})}]}

In [18]:
import pandas as pd
from tabulate import tabulate

entities = pd.read_parquet('./output/entities.parquet')

# 将数组类型的列转为字符串
df_display = entities.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+---------------------+--------------+----------------------+----------------------+-----------+--------+
| id                   | human_readable_id | title               | type         | description          | text_unit_ids        | frequency | degree |
+----------------------+-------------------+---------------------+--------------+----------------------+----------------------+-----------+--------+
| d48c208c-de2f-4bd2-a | 0                 | HUAWEI              | ORGANIZATION | Huawei is a global   | ['3755a97bbf1d29f73d | 2         | 7      |
| 22a-171f6c509ed4     |                   |                     |              | leader in            | a6033a6fc991e50fd5a2 |           |        |
|                      |                   |                     |              | information and      | ea859a8f37dd7f98fbb4 |           |        |
|                      |                   |                     |              | communication        | 8

In [19]:
def setup_entity_constraints():
    """创建entity标签的约束"""
    with driver.session() as session:
        try:
            # 创建Entity.id唯一性约束
            session.run("create constraint if not exists for (e:__Entity__) require e.id is unique")
            session.run("create constraint if not exists for (e:__Entity__) require e.name is unique")
            print("已创建__Entity__.id唯一性约束")
        except Exception as e:
            print(f"创建__Entity__约束时出错（可能已存在）：{e}")
            # 尝试旧版本Neo4j的语法
            try:
                session.run("create constraint on (e:__Entity__) assert e.id is unique")
                print("已使用旧语法创建__Entity__.id唯一性约束")
            except Exception as e1:
                print(f"使用旧语法创建约束也失败：{e1}")

def import_entities(df_entities,batch_size=100,max_workers=8):
    """
    导入实体（Entity）到Neo4j
    
    参数：
    - df_entities: 包含实体数据的DataFrame
    - batch_size: 每批处理的行数
    - max_workers: 并行线程数
    
    返回：
    - 导入统计信息的字典
    """
    
    # 1. 创建Entity的约束
    setup_entity_constraints()
    
    # 2. 预处理text_unit_ids - 确保是列表格式
    print("预处理text_unit_ids...")
    
    # 创建DataFrame的副本
    df_entities=df_entities.copy()
    
    # 处理text_unit_ids 字段
    for idx, row in df_entities.iterrows():
        text_unit_ids=row.get('text_unit_ids')

        # 如果不是列表，转换为列表
        if not isinstance(text_unit_ids,list):
            if isinstance(text_unit_ids,str):
                try:
                    # 尝试解析JSON字符串
                    import json
                    text_unit_ids=json.loads(text_unit_ids)
                except:
                    # 如果解析失败，将其作为单个元素的列表
                    text_unit_ids=[text_unit_ids]
            elif hasattr(text_unit_ids,'dtype') and hasattr(text_unit_ids,'tolist'):
                # 处理NumPy数组
                text_unit_ids=text_unit_ids.tolist()
            else:
                # 其他类型，转为列表
                text_unit_ids=[text_unit_ids] if text_unit_ids is not None else []
        
        # 处理嵌套列表
        flat_text_unit_ids=[]
        for item in text_unit_ids:
            if isinstance(item,list) or (hasattr(item, 'dtype') and hasattr(item,'tolist')):
                if hasattr(item,'tolist'):
                    flat_text_unit_ids.extend(item.tolist())
                else:
                    flat_text_unit_ids.extend(item)
            else:
                flat_text_unit_ids.append(item)
        
        # 确保所有ID都是字符串且非空
        flat_text_unit_ids=[str(id) for id in flat_text_unit_ids if id is not None and str(id).strip()!='']
        
        # 更新DataFrame
        df_entities.at[idx,'text_unit_ids']=flat_text_unit_ids
    
    # 3. 检查Neo4j功能支持
    print("检查Neo4j功能支持...")
    has_apoc=False
    has_vector=False
    
    try:
        with driver.session() as session:
            # 检查APOC，我们目前使用的不是插件模式
            try:
                result=session.run("return apoc.version() as version")
                version=result.single()['version']
                has_apoc=True
                print(f"APOC插件已安装，版本：{version}")
            except Exception as e:
                print(f"检查APOC插件时出错(可能未安装): {e}")
    except Exception as e:
        print(f"检查Neo4j功能支持时出错：{e}")
        
    # 4. 导入entity节点并创建关系
    print("开始导入__Entity__节点并创建关系...")
    
    # 根据功能支持构建Cypher语句
    if has_apoc and has_vector:
        # 完整功能支持
        entity_statement="""
        merge (e:__Entity__{id:value.id})
        set e+=value {.human_readable_id,.description,.frequency,.degree,.x,.y}
        set e.name=replace(coalesce(value.title,value.human_readable_id,''),'"','')
        
        with e,value
        call db.create.setNodeVectorProperty(e,"description_embedding",value.description_embedding)
        
        with e,value
        call apoc.create.addLabels(e,
        case when coalesce(value.type,"")=""
        then []
        else [apoc.text.upperCamelCase(replace(value.type,'"',''))]
        end
        ) yield node
        
        with node as e, value
        unwind value.text_unit_ids as text_unit
        match (c:__Chunk__{id:text_unit})
        merge (c)-[:HAS_ENTITY]->(e)
        """
    elif has_vector:
        # 只有向量支持，没有APOC支持
        entity_statement="""
        merge (e:__Entity__{id:value.id})
        set e+=value {.human_readable_id,.description,.frequency,.degree,.x,.y}
        set e.name=replace(coalesce(value.title,value.human_readable_id,''),'"','')
        
        with e,value
        call db.create.setNodeVectorProperty(e,"description_embedding",value.description_embedding)
        
        with e, value
        unwind value.text_unit_ids as text_unit
        match (c:__Chunk__{id:text_unit})
        merge (c)-[:HAS_ENTITY]->(e)
        """
    else:
        # 基本功能，无APOC和向量支持
        entity_statement="""
        merge (e:__Entity__{id:value.id})
        set e+=value {.human_readable_id,.description,.frequency,.degree,.x,.y}
        set e.name=replace(coalesce(value.title,value.human_readable_id,''),'"','')
        
        with e, value
        unwind value.text_unit_ids as text_unit
        match (c:__Chunk__{id:text_unit})
        merge (c)-[:HAS_ENTITY]->(e)
        """
        
    # 执行导入
    entity_result=parallel_batched_import(entity_statement,df_entities,batch_size,max_workers)
    
    # 5. 验证结果
    with driver.session() as session:
        # 检查Entity节点数量
        result=session.run("match (e:__Entity__) return count(e) as count")
        entity_count=result.single()['count']
        
        # 检查HAS_ENTITY关系数量
        result=session.run("match (c:__Chunk__)-[r:HAS_ENTITY]->() return count(r) as count")
        relation_count=result.single()['count']
        
        # 检查标签
        result=session.run("call db.labels() yield label where label <> '__ENTITY__' and label <> '__Chunk__' return collect(label) as labels")
        dynamic_labels=result.single()['labels']
        
        print(f"验证结果：{entity_count} 个__ENTITY__节点，{relation_count} 个HAS_ENTITY关系")
        print(f"动态标签：{dynamic_labels}")
        
    return entity_result
    
    
                
    

In [20]:
import_entities(entities,batch_size=100,max_workers=8)

已创建__Entity__.id唯一性约束
预处理text_unit_ids...
检查Neo4j功能支持...
检查APOC插件时出错(可能未安装): {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Unknown function 'apoc.version' (line 1, column 8 (offset: 7))
"return apoc.version() as version"
        ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}
开始导入__Entity__节点并创建关系...
开始并行导入 29 行数据，分为1 个批次，每批100 条
批次 0 完成：29 行，耗时：0.10秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 29 行，成功29 行，失败0行
总耗时：0.11秒，平均速度:269.52行/秒
验证结果：29 个__ENTITY__节点，34 个HAS_ENTITY关系
动态标签：['Person', 'Company', 'Student', '__Document__', '__Entity__']


{'total_rows': 29,
 'successful_rows': 29,
 'failed_rows': 0,
 'duration_seconds': 0.10760068893432617,
 'rows_per_second': 269.515002991292,
 'batch_results': [{'batch': 0,
   'rows': 29,
   'success': True,
   'duration': 0.10443782806396484,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 29, 'relationships-created': 34, 'nodes-created': 29, 'properties-set': 174})}]}

In [21]:
import pandas as pd
from tabulate import tabulate

relationships = pd.read_parquet('./output/relationships.parquet')

# 将数组类型的列转为字符串
df_display = relationships.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+---------------+---------------------+----------------------+--------+-----------------+----------------------+
| id                   | human_readable_id | source        | target              | description          | weight | combined_degree | text_unit_ids        |
+----------------------+-------------------+---------------+---------------------+----------------------+--------+-----------------+----------------------+
| 48f869c3-145d-44f1-8 | 0                 | HUAWEI        | SHENZHEN            | Huawei was founded   | 9.0    | 8               | ['3755a97bbf1d29f73d |
| 656-7e654f18226f     |                   |               |                     | in Shenzhen in 1987  |        |                 | a6033a6fc991e50fd5a2 |
|                      |                   |               |                     |                      |        |                 | ea859a8f37dd7f98fbb4 |
|                      |                   |               |    

In [29]:
def setup_relationships_constraints():
    """创建Relationship标签的约束"""
    with driver.session() as session:
        try:
            # 创建Relationship.id唯一性约束
            session.run("create constraint if not exists for (r:__Relationship__) require r.id is unique")
            print("已创建__Relationship__.id唯一性约束")
        except Exception as e:
            print(f"创建__Relationship__约束时出错(可能已存在):{e}")
            # 尝试旧版本Neo4j的语法
            try:
                session.run("Create constraint on (r:__Relationship__) assert r.id is unique")
                print("已创建__Relationship__.id唯一性约束")
            except Exception as e2:
                print(f"使用旧语法创建约束也失败： {e2}")
def import_relationships(df_relaiontships,batch_size=100,max_workers=8):
    """
    导入关系数据到Neo4j
    
    参数：
    
    - df_relaiontships: 包含关系数据的DataFrame
    - batch_size: 每批处理的行数
    - max_workers: 并行线程数
    
    返回:
    - 导入统计信息的字典
    """
    
    # 1. 创建Relationship的约束
    setup_relationships_constraints()
    
    # 2. 检查标签和节点是否存在
    with driver.session() as session:
        # 检查entity和chunk节点数量
        result=session.run("match (e:__Entity__) return count(e) as count")
        entity_count=result.single()['count']
        
        result=session.run("match (c:__Chunk__) return count(c) as count")
        chunk_count=result.single()['count']
        
        print(f"节点数量： {entity_count}个__Entity__节点，{chunk_count}个__Chunk__节点")
        
        if entity_count==0:
            print("没有__Entity__节点，请先导入Entity数据")
        if chunk_count==0:
            print("没有__Chunk__节点，请先导入Chunk数据")
        
    # 3. 预处理预处理text_unit_ids - 确保是列表格式
    print("预处理text_unit_ids...")
    
    # 创建DataFrame的副本
    df_relaiontships=df_relaiontships.copy()
    
    # 处理text_unit_ids字段
    for idx,row in df_relaiontships.iterrows():
        text_unit_ids=row.get("text_unit_ids")
        
        # 如果不是列表，转换为列表
        if not isinstance(text_unit_ids,list):
            if isinstance(text_unit_ids,str):
                try:
                    # 尝试解析JSON字符串
                    import json
                    text_unit_ids=json.loads(text_unit_ids)
                except:
                    # 如果解析失败，将其作为单个元素的列表
                    text_unit_ids=[text_unit_ids]
            elif hasattr(text_unit_ids,'dtype') and hasattr(text_unit_ids,'tolist'):
                # 处理NumPy数组
                text_unit_ids=text_unit_ids.tolist()
            else:
                # 其他类型，转为列表
                text_unit_ids=[text_unit_ids] if text_unit_ids is not None else []
        # 处理嵌套列表
        flat_text_unit_ids=[]
        for item in text_unit_ids:
            if isinstance(item,list) or (hasattr(item,'dtype') and hasattr(item,'tolist')):
                if hasattr(item,'tolist'):
                    flat_text_unit_ids.extend(item.tolist())
                else:
                    flat_text_unit_ids.extend(item)
            else:
                flat_text_unit_ids.append(item)
        
        
        # 确保所有ID都是字符串且非空
        flat_text_unit_ids=[str(id) for id in flat_text_unit_ids if id is not None and str(id).strip() !='']

        #更新DataFrame
        df_relaiontships.at[idx,'text_unit_ids']=flat_text_unit_ids
        
    # 4. 验证source和target是否存在
    print("验证source和target实体...")
    sample_rows=df_relaiontships.head(min(5,len(df_relaiontships))) #  取前五行验证
    
    with driver.session() as session:
        for _,row in sample_rows.iterrows():
            source_id=row['source']
            target_id=row['target']
            
            # 检查source实体
            result=session.run("match (e:__Entity__{id:$id}) return count(e) as count",id=source_id)
            source_exists=result.single()['count']>0
            
            # 检查target实体
            result=session.run("match (e:__Entity__{id:$id}) return count(e) as count",id=target_id)
            target_exists=result.single()['count']>0
        
            print(f"实体检查： source={source_id} {'存在' if source_exists else '不存在'}, target={target_id} {'存在' if target_exists else '不存在'}")

            if not source_exists or not target_exists:
                print("部分实体不存在，将在导入过程中创建")
    # 5. 导入关系数据
    print("开始导入关系数据...")
    
    # 分两步执行，先创建关系节点和实体间关系，再创建与chunk的关系
    # 步骤1： 创建关系节点和实体间的关系
    relationship_statement="""
    //创建关系节点
    
    merge (r:__Relationship__{id:value.id})
    set r.human_readable_id=value.human_readable_id,
    r.description=value.description,
    r.weight=value.weight,
    r.combined_degree=value.combined_degree,
    r.name=value.human_readable_id
    
    //查找或创建源实体和目标实体
    with r, value
    merge (source:__Entity__{id:value.source})
    merge (target:__Entity__{id:value.target})
    
    //创建RELATED关系
    merge (source)-[rel:RELATED]->(target)
    set rel.relationship_id=value.id,
    rel.description=value.description,
    rel.weight=value.weight
    
    return r.id as relationship_id
    """
    # 执行导入关系节点和实体间关系
    relationship_result=parallel_batched_import(relationship_statement,df_relaiontships,batch_size,max_workers)
    print(f"已创建{relationship_result['successful_rows']}个__Relationship__节点和RELATED关系")
    
    # 步骤2： 创建与Chunk的关系
    # 准备Chunk关系数据
    chunk_relations=[]
    for _,row in df_relaiontships.iterrows():
        rel_id=row['id']
        for chunk_id in row['text_unit_ids']:
            chunk_relations.append({
                'relationship_id':rel_id,
                'chunk_id':chunk_id
            })
    
    if chunk_relations:
        df_chunk_relations=pd.DataFrame(chunk_relations)
        
        # 创建Chunk和Relationship的关系
        chunk_rel_statement="""
        match (r:__Relationship__{id:value.relationship_id})
        match (c:__Chunk__{id:value.chunk_id})
        merge (c)-[:HAS_RELATIONSHIP]->(r)
        """
        
        # 执行导入chunk关系
        chunk_rel_result=parallel_batched_import(chunk_rel_statement,df_chunk_relations,batch_size,max_workers)
        print(f"已创建 {chunk_rel_result['successful_rows']}个Chunk-Relationship关系")
    else:
        print("没有找到有效的Chunk-Relationship关系数据")

        
    # 6. 验证结果
    with driver.session() as session:
        # 检查Relationship节点数量
        result=session.run("match (r:__Relationship__) return count(r) as count")
        relationship_count=result.single()['count']
        
        # 检查RELATED关系数量
        result=session.run("match ()-[r:RELATED]->() return count(r) as count")
        related_count=result.single()['count']
        
        # 检查HAS_RELATIONSHIP关系数量
        try:
            result=session.run("match (c:__Chunk__)-[r:HAS_RELATIONSHIP]->() return count(r) as count")
            has_relationship_count=result.single()['count']
        except Exception as e:
            print(f"查询HAS_RELATIONSHIP关系时出错: {e}")
            has_relationship_count="未知"
        
        print(f"验证结果 {relationship_count} 个__Relationship__节点，{related_count}个RELATED关系,{has_relationship_count}个HAS_RELATIONSHIP关系")
        
    return relationship_result

In [30]:
import_relationships(relationships,batch_size=100,max_workers=8)

已创建__Relationship__.id唯一性约束
节点数量： 29个__Entity__节点，3个__Chunk__节点
预处理text_unit_ids...
验证source和target实体...
实体检查： source=HUAWEI 不存在, target=SHENZHEN 不存在
部分实体不存在，将在导入过程中创建
实体检查： source=HUAWEI 不存在, target=HARMONYOS 不存在
部分实体不存在，将在导入过程中创建
实体检查： source=HUAWEI 不存在, target=SMART ECOLOGY 不存在
部分实体不存在，将在导入过程中创建
实体检查： source=HUAWEI 不存在, target=ICT 不存在
部分实体不存在，将在导入过程中创建
实体检查： source=XIAOMI 不存在, target=LEI JUN 不存在
部分实体不存在，将在导入过程中创建
开始导入关系数据...
开始并行导入 28 行数据，分为1 个批次，每批100 条
批次 0 完成：28 行，耗时：0.45秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 28 行，成功28 行，失败0行
总耗时：0.47秒，平均速度:59.15行/秒
已创建28个__Relationship__节点和RELATED关系
开始并行导入 35 行数据，分为1 个批次，每批100 条
批次 0 完成：35 行，耗时：0.10秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 35 行，成功35 行，失败0行
总耗时：0.10秒，平均速度:349.69行/秒
已创建 35个Chunk-Relationship关系
验证结果 28 个__Relationship__节点，28个RELATED关系,29个HAS_RELATIONSHIP关系


{'total_rows': 28,
 'successful_rows': 28,
 'failed_rows': 0,
 'duration_seconds': 0.4733612537384033,
 'rows_per_second': 59.15144042497788,
 'batch_results': [{'batch': 0,
   'rows': 28,
   'success': True,
   'duration': 0.4521298408508301,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 54, 'relationships-created': 28, 'nodes-created': 54, 'properties-set': 278})}]}

In [31]:
import pandas as pd
from tabulate import tabulate

communities = pd.read_parquet('./output/communities.parquet')

# 将数组类型的列转为字符串
df_display = communities.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+-----------+-------+--------+----------+-------------+----------------------+----------------------+----------------------+------------+------+
| id                   | human_readable_id | community | level | parent | children | title       | entity_ids           | relationship_ids     | text_unit_ids        | period     | size |
+----------------------+-------------------+-----------+-------+--------+----------+-------------+----------------------+----------------------+----------------------+------------+------+
| 7be7e379-a7f7-4d72-a | 0                 | 0         | 0     | -1     | [3 4]    | Community 0 | ['a6167453-c23a-45cc | ['19a7ad6f-5112-4974 | ['b366e0fc7d595bdbb9 | 2026-07-09 | 10   |
| 9e2-bce7e5beae9d     |                   |           |       |        |          |             | -a6d9-fb85f6c4924e'  | -8fae-c5c0e97fccd0'  | 06263a4f2670b2310998 |            |      |
|                      |                   |           |    

In [32]:
def setup_community_constraints():
    """创建Community标签的约束"""
    with driver.session() as session:
        try:
            # 创建Relationship.id唯一性约束
            session.run("create constraint if not exists for (r:__Community__) require r.id is unique")
            print("已创建__Community__.id唯一性约束")
        except Exception as e:
            print(f"创建__Community__约束时出错(可能已存在):{e}")
            # 尝试旧版本Neo4j的语法
            try:
                session.run("Create constraint on (r:__Community__) assert r.id is unique")
                print("已创建__Community__.id唯一性约束")
            except Exception as e2:
                print(f"使用旧语法创建约束也失败： {e2}")
def import_communities(df_communities,batch_size=100,max_workers=8):
    """
    导入社区(Community)数据到Neo4j
    
    参数：
    
    - df_communities: 包含社区关系数据的DataFrame
    - batch_size: 每批处理的行数
    - max_workers: 并行线程数
    
    返回:
    - 导入统计信息的字典
    """
    
    # 1. 创建Community的约束
    setup_community_constraints()
    
    # 2. 预处理列表字段 - 确保是列表格式
    print("预处理列表字段...")
    
    # 创建DataFrame的副本
    df_communities=df_communities.copy()
    
    # 需要处理的列表字段
    list_fields=['children','entity_ids','relationship_ids','text_unit_ids']
    
    for field in list_fields:
        if field in df_communities.columns:
            for idx,row in df_communities.iterrows():
                field_value=row.get(field)
                
                # 如果不是列表，转换为列表
                if not isinstance(field_value,list):
                    if isinstance(field_value,str):
                        try:
                            # 尝试解析JSON字符串
                            import json
                            field_value=json.loads(field_value)
                        except:
                            # 如果解析失败，将其作为单个元素的列表
                            field_value=[field_value]
                    elif hasattr(field_value,'dtype') and hasattr(field_value,'tolist'):
                        # 处理NumPy数组
                        field_value=field_value.tolist()
                    else:
                        # 其他类型，转为列表
                        field_value=[field_value] if field_value is not None else []
                # 处理嵌套列表
                flat_field_value=[]
                for item in field_value:
                    if isinstance(item,list) or (hasattr(item,'dtype') and hasattr(item,'tolist')):
                        if hasattr(item,'tolist'):
                            flat_field_value.extend(item.tolist())
                        else:
                            flat_field_value.extend(item)
                    else:
                        flat_field_value.append(item)
        
    # 3. 导入社区节点
    print("开始导入社区节点...")
    
    community_statement="""
    //创建community节点
    
    merge (c:__Community__{id:value.id})
    set c.human_readable_id=value.human_readable_id,
    c.community=value.community,
    c.level=value.level,
    c.parent=value.parent,
    c.children=value.children,
    c.title=value.title,
    c.period=value.period,
    c.size=value.size,
    c.name=coalesce(value.title,value.human_readable_id,'Community_'+value.id)
    
    return c.id as community_id
    """
    # 执行导入社区节点
    community_result=parallel_batched_import(community_statement,df_communities,batch_size,max_workers)
    print(f"已创建{community_result['successful_rows']}个__Community__节点")
    
    # 4. 创建与Entity关系
    print("开始创建社区与实体的关系")
    
    # 准备Entity 关系数据
    entity_relations=[]
    
    for _,row in df_communities.iterrows():
        community_id=row['id']
        entity_ids=row.get('entity_ids',[])
        
        for entity_id in entity_ids:
            entity_relations.append({
                'community_id':community_id,
                'entity_id':entity_id
            })
    
    if entity_relations:
        df_entity_relations=pd.DataFrame(entity_relations)
        
        entity_rel_statement="""
        match (c:__Community__{id:value.community_id})
        match (e:__Entity__{id:value.entity_id})
        merge (e)-[:IN_COMMUNITY]->(c)
        """
        
        entity_rel_result=parallel_batched_import(entity_rel_statement,df_entity_relations,batch_size,max_workers)
        print(f"已创建 {entity_rel_result['successful_rows']}个Entity-Community关系")
    else:
        print("没有找到有效的Entity-Community关系数据")
        
    # 5. 创建与Relationship的关系
    print("开始创建社区与关系的关系")
    
    # 准备Relationship关系数据
    rel_relations=[]
    for _,row in df_communities.iterrows():
        community_id=row['id']
        relationship_ids=row.get('relationship_ids',[])
        
        for rel_id in relationship_ids:
            rel_relations.append({
                'community_id':community_id,
                'relationship_id':rel_id
            })
    
    if rel_relations:
        df_rel_relations=pd.DataFrame(rel_relations)
        
        rel_rel_statement="""
        match (c:__Community__{id:value.community_id})
        match (r:__Relationship__{id:value.relationship_id})
        merge (r)-[:IN_COMMUNITY]->(c)
        """
        
        rel_rel_result=parallel_batched_import(rel_rel_statement,df_rel_relations,batch_size,max_workers)
        print(f"已创建{rel_rel_result['successful_rows']} 个Relationship-Community关系")
    else:
        print("没有找到有效的Relationship-Community关系")
        
    # 6. 验证结果
    with driver.session() as session:
        # 检查Community节点数量
        result= session.run("match (c:__Community__) return count(c) as count")
        community_count=result.single()['count']
        
        # 检查IN_COMMUNITY关系数量
        result=session.run("match ()-[r:IN_COMMUNITY]->() return count(r) as count")
        in_community_count=result.single()['count']
        
        print(f"验证结果： {community_count}个__Community__节点，{in_community_count}个IN_COMMUNITY关系")
    
    return community_result

In [33]:
import_communities(communities,batch_size=100,max_workers=8)

已创建__Community__.id唯一性约束
预处理列表字段...
开始导入社区节点...
开始并行导入 5 行数据，分为1 个批次，每批100 条
批次 0 完成：5 行，耗时：0.15秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 5 行，成功5 行，失败0行
总耗时：0.15秒，平均速度:33.86行/秒
已创建5个__Community__节点
开始创建社区与实体的关系
开始并行导入 36 行数据，分为1 个批次，每批100 条
批次 0 完成：36 行，耗时：0.11秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 36 行，成功36 行，失败0行
总耗时：0.11秒，平均速度:320.28行/秒
已创建 36个Entity-Community关系
开始创建社区与关系的关系
开始并行导入 32 行数据，分为1 个批次，每批100 条
批次 0 完成：32 行，耗时：0.08秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 32 行，成功32 行，失败0行
总耗时：0.08秒，平均速度:408.65行/秒
已创建32 个Relationship-Community关系
验证结果： 5个__Community__节点，68个IN_COMMUNITY关系


{'total_rows': 5,
 'successful_rows': 5,
 'failed_rows': 0,
 'duration_seconds': 0.14768242835998535,
 'rows_per_second': 33.85643136780078,
 'batch_results': [{'batch': 0,
   'rows': 5,
   'success': True,
   'duration': 0.14501547813415527,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 5, 'nodes-created': 5, 'properties-set': 50})}]}